# ***Notebook For Standardizing Tables For Logistic DB***

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
import os
from pathlib import Path

from sqlalchemy import create_engine, text

import warnings
warnings.filterwarnings('ignore')

In [2]:
env_path = Path.cwd()/"github_repo_cloned"/"the_distribution_hub_optimizer"/".env"
load_dotenv(env_path)


host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
paswd = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")
output_dir = Path(os.getenv("OUTPUT_DIR", Path.cwd()/"outputs"))
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
engine = create_engine(f"mysql+pymysql://{user}:{paswd}@{host}:{port}/{db}")


def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df

In [4]:
bill = sql('select * from bill_raw') 

In [5]:
gsdp = sql('select * from gsdp_raw') 

In [6]:
population = sql('select * from population_raw')

## **Data Standardization**

In [7]:
bill.head()

,state_code,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset
0,1,JAMMU AND KASHMIR,2018,7,3171,120989,2083.7662087950002,1711,25744,2243.508837,15449,151999,4567.124618
1,1,JAMMU AND KASHMIR,2018,8,3292,126112,2245.7754784900003,1996,30376,2535.885482,17023,180173,5157.642880
2,1,JAMMU AND KASHMIR,2018,9,3484,130396,2387.847554812,2114,30149,2215.296839,16962,186069,5240.539348
3,1,JAMMU AND KASHMIR,2018,10,3784,135435,2545.633385386,2273,32807,2160.836647,17722,202717,5138.989032
4,1,JAMMU AND KASHMIR,2018,11,3851,123745,2178.749353559,2080,26704,1751.514975,16757,165151,4435.449635


In [8]:
gsdp.head()

,state/ut,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24
0,Andhra Pradesh,873721.106301,925839.118118,978581.462125,1.148471e+06,1.303524e+06,1439673.87
1,Arunachal Pradesh,25334.870000,30023.650000,30525.350000,3.477476e+04,3.962996e+04,43991.61
2,Assam,309336.320000,346850.680000,339802.980000,4.114538e+05,4.931666e+05,565400.58
3,Bihar,527975.819958,581855.484686,567262.504733,6.503024e+05,7.513956e+05,854429.00
4,Chhattisgarh,327106.660000,344672.040000,352327.510000,4.105249e+05,4.643990e+05,505886.51


In [9]:
population.head()

,state/ut,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,Jammu & Kashmir*(UT),13101,13203,13305,13408,13505,13603,13701,13798,13896
1,Himachal Pradesh,7253,7300,7347,7394,7431,7468,7505,7542,7579
2,Punjab,29619,29859,30099,30339,30535,30730,30926,31122,31318
3,Haryana,28266,28672,29077,29483,29846,30209,30573,30936,31299
4,NCT of Delhi*,19435,19814,20193,20571,20965,21359,21752,22146,22540


### **Initial Integrity Check - Obvious Errors**

#### **Bill Table**

In [10]:
bill.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3664 entries, 0 to 3663
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   state_code           3664 non-null   int64  
 1   state_name           3664 non-null   object 
 2   year                 3664 non-null   int64  
 3   month                3664 non-null   int64  
 4   intra_suplliers      3664 non-null   object 
 5   intra_bill           3664 non-null   object 
 6   intra_assets         3664 non-null   object 
 7   inter_out_suplliers  3664 non-null   int64  
 8   inter_out_bill       3664 non-null   int64  
 9   inter_out_asset      3664 non-null   float64
 10  inter_in_suplliers   3664 non-null   int64  
 11  inter_in_bill        3664 non-null   int64  
 12  inter_in_asset       3664 non-null   float64
dtypes: float64(2), int64(7), object(4)
memory usage: 372.3+ KB


In [11]:
bill.columns

Index(['state_code', 'state_name', 'year', 'month', 'intra_suplliers',
       'intra_bill', 'intra_assets', 'inter_out_suplliers', 'inter_out_bill',
       'inter_out_asset', 'inter_in_suplliers', 'inter_in_bill',
       'inter_in_asset'],
      dtype='object')

`Fixing column data type`

In [12]:
bill = bill.replace("-",np.nan)

In [13]:
whole_cols = ['intra_suplliers', 'intra_bill',]
float_cols = [ 'intra_assets']
bill[float_cols] = bill[float_cols].astype(float)
bill[whole_cols] = bill[whole_cols].astype("Int64")

```Summary Of Numerical Columns```

In [14]:
actual_numeric = ['intra_suplliers',
       'intra_bill', 'intra_assets', 'inter_out_suplliers', 'inter_out_bill',
       'inter_out_asset', 'inter_in_suplliers', 'inter_in_bill',
       'inter_in_asset']


bill[actual_numeric].describe().round(2).T[['min','50%','max','std','mean']].rename({'50%':'median'},axis = 1)

,min,median,max,std,mean
intra_suplliers,0.0,9260.5,224042.0,45944.15,31829.32
intra_bill,0.0,355588.5,15722516.0,2216865.04,1353582.14
intra_assets,0.0,7614.0,351408.1,46162.9,28266.5
inter_out_suplliers,0.0,6697.0,168901.0,37773.49,25316.82
inter_out_bill,0.0,195438.0,8649985.0,1336783.14,771196.64
inter_out_asset,0.0,9506.45,254765.99,39191.2,26511.34
inter_in_suplliers,5.0,24052.5,259235.0,56359.0,47376.21
inter_in_bill,20.0,356606.5,5399958.0,987534.15,768406.13
inter_in_asset,0.38,12866.7,259141.83,37841.94,27908.82


---
3 columns are showing minimum value as 0, might be an issue in the data, worth checking

---

`Checking rows where value is 0 in any columns`

In [15]:
zero_cols = [ 'inter_out_suplliers', 'inter_out_bill','inter_out_asset', ]

print(f"Rows with 0 as values in any column: \n{(bill[zero_cols]==0).sum()}")

print("Rows with 0 values: ")
bill[(bill[zero_cols]==0).any(axis=1)].sort_values(by = ['state_name','year','month'],ascending=True)

Rows with 0 as values in any column: 
inter_out_suplliers    45
inter_out_bill         45
inter_out_asset        47
dtype: int64
Rows with 0 values: 


,state_code,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset
2366,25,DAMAN AND DIU,2021,10,0,0,0.000000,0,0,0.0,10,22,0.383400
2362,25,DAMAN AND DIU,2022,2,0,0,0.000000,0,0,0.0,19,60,2.086800
2375,25,DAMAN AND DIU,2023,1,0,0,0.000000,0,0,0.0,20,25,0.988600
2396,25,DAMAN AND DIU,2023,4,0,0,0.000000,0,0,0.0,11,20,0.596800
2395,25,DAMAN AND DIU,2023,5,0,0,0.000000,0,0,0.0,13,20,0.849900
2394,25,DAMAN AND DIU,2023,6,0,0,0.000000,0,0,0.0,21,33,2.185500
2393,25,DAMAN AND DIU,2023,7,0,0,0.000000,0,0,0.0,15,28,1.435300
2388,25,DAMAN AND DIU,2023,12,0,0,0.000000,0,0,0.0,25,46,1.515900
2387,25,DAMAN AND DIU,2024,1,0,0,0.000000,0,0,0.0,24,29,0.770900
2386,25,DAMAN AND DIU,2024,2,0,0,0.000000,0,0,0.0,23,39,1.582900


---
0's are only from lakshadeep and daman and diu, since daman diu merged with dadar and nagar haveli, I will treat those 2 states as same state from the year 2019, for the sake of this analysis, 

and technically - **no company would want to open a hub in island like lakshadeep or andamans either, I will remove those 2 islands as well**

---

`Adding Dadar and Daman Diu together from the year 2019 onwards`

In [16]:
bill['state_name'].unique()

array([' JAMMU AND KASHMIR', 'JAMMU AND KASHMIR', ' HIMACHAL PRADESH',
       'HIMACHAL PRADESH', ' PUNJAB', 'PUNJAB', ' CHANDIGARH',
       'CHANDIGARH', ' UTTARAKHAND', 'UTTARAKHAND', ' HARYANA', 'HARYANA',
       ' DELHI', 'DELHI', ' RAJASTHAN', 'RAJASTHAN', ' UTTAR PRADESH',
       'UTTAR PRADESH', ' BIHAR', 'BIHAR', ' SIKKIM', 'SIKKIM',
       ' ARUNACHAL PRADESH', 'ARUNACHAL PRADESH', ' NAGALAND', 'NAGALAND',
       ' MANIPUR', 'MANIPUR', ' MIZORAM', 'MIZORAM', ' TRIPURA',
       'TRIPURA', ' MEGHALAYA', 'MEGHALAYA', ' ASSAM', 'ASSAM',
       ' WEST BENGAL', 'WEST BENGAL', ' JHARKHAND', 'JHARKHAND',
       ' ODISHA', 'ODISHA', ' CHHATTISGARH', 'CHHATTISGARH',
       ' MADHYA PRADESH', 'MADHYA PRADESH', ' GUJARAT', 'GUJARAT',
       ' DAMAN AND DIU', 'DAMAN AND DIU', ' DADRA AND NAGAR HAVELI',
       'DADRA AND NAGAR HAVELI', ' MAHARASHTRA', 'MAHARASHTRA',
       ' KARNATAKA', 'KARNATAKA', ' GOA', 'GOA', ' LAKSHADWEEP',
       'LAKSHADWEEP', ' KERALA', 'KERALA', ' TAMIL NADU', 'TA

---
There are duplicacy in column names, with leading and trailling whitespaces, need them to be removed

---

`Removing spaces from state names`

In [17]:
bill['state_name'] = bill['state_name'].str.strip()

In [18]:
bill['state_name'].unique()

array(['JAMMU AND KASHMIR', 'HIMACHAL PRADESH', 'PUNJAB', 'CHANDIGARH',
       'UTTARAKHAND', 'HARYANA', 'DELHI', 'RAJASTHAN', 'UTTAR PRADESH',
       'BIHAR', 'SIKKIM', 'ARUNACHAL PRADESH', 'NAGALAND', 'MANIPUR',
       'MIZORAM', 'TRIPURA', 'MEGHALAYA', 'ASSAM', 'WEST BENGAL',
       'JHARKHAND', 'ODISHA', 'CHHATTISGARH', 'MADHYA PRADESH', 'GUJARAT',
       'DAMAN AND DIU', 'DADRA AND NAGAR HAVELI', 'MAHARASHTRA',
       'KARNATAKA', 'GOA', 'LAKSHADWEEP', 'KERALA', 'TAMIL NADU',
       'PUDUCHERRY', 'ANDAMAN AND NICOBAR', 'TELANGANA', 'ANDHRA PRADESH',
       'LADAKH', 'Other Territory'], dtype=object)

`Merging 2 states, to act as one`

In [19]:
bill.drop('state_code',axis=1,inplace=True)

In [20]:
states_to_merge = ['DADRA AND NAGAR HAVELI','DAMAN AND DIU']
new_name = 'DADRA NAGAR HAVELI & DAMAN DIU'

bill['state_name'] = bill['state_name'].replace(states_to_merge,new_name)


bill = bill.groupby(['state_name','year','month'],as_index=False)[actual_numeric].sum()

`Removing the islands and Other territory`

In [21]:
bill['state_name'].unique()

array(['ANDAMAN AND NICOBAR', 'ANDHRA PRADESH', 'ARUNACHAL PRADESH',
       'ASSAM', 'BIHAR', 'CHANDIGARH', 'CHHATTISGARH',
       'DADRA NAGAR HAVELI & DAMAN DIU', 'DELHI', 'GOA', 'GUJARAT',
       'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND',
       'KARNATAKA', 'KERALA', 'LADAKH', 'LAKSHADWEEP', 'MADHYA PRADESH',
       'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'NAGALAND',
       'ODISHA', 'Other Territory', 'PUDUCHERRY', 'PUNJAB', 'RAJASTHAN',
       'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH',
       'UTTARAKHAND', 'WEST BENGAL'], dtype=object)

In [22]:
remove_list = ['LAKSHADWEEP', 'ANDAMAN AND NICOBAR', 'Other Territory']

bill = bill[~bill['state_name'].isin(remove_list)]

---
Removing and Merging of problematic states are done here

---

`Check the count of each state, how many months of values it have, and what is the minumum and maximum year for all`

In [23]:
bill.groupby('state_name')['month'].count().reset_index(name = 'counts')

,state_name,counts
0,ANDHRA PRADESH,97
1,ARUNACHAL PRADESH,97
2,ASSAM,97
3,BIHAR,97
4,CHANDIGARH,97
5,CHHATTISGARH,97
6,DADRA NAGAR HAVELI & DAMAN DIU,97
7,DELHI,97
8,GOA,97
9,GUJARAT,97


In [24]:
bill['date'] = pd.to_datetime(bill[['year', 'month']].assign(day=1))

bill.groupby('state_name',as_index=False)['date'].agg(['min', 'max', 'count'])

,state_name,min,max,count
0,ANDHRA PRADESH,2018-07-01,2026-07-01,97
1,ARUNACHAL PRADESH,2018-07-01,2026-07-01,97
2,ASSAM,2018-07-01,2026-07-01,97
3,BIHAR,2018-07-01,2026-07-01,97
4,CHANDIGARH,2018-07-01,2026-07-01,97
5,CHHATTISGARH,2018-07-01,2026-07-01,97
6,DADRA NAGAR HAVELI & DAMAN DIU,2018-07-01,2026-07-01,97
7,DELHI,2018-07-01,2026-07-01,97
8,GOA,2018-07-01,2026-07-01,97
9,GUJARAT,2018-07-01,2026-07-01,97


---
Ladakh have less months of data than other months, and it is also remote location for a hub..safe to remove from the data

---

`Removing ladakh`

In [25]:
bill = bill[bill['state_name'] != 'LADAKH']

`Checking the stats after state cleaning`

In [26]:
actual_numeric = ['intra_suplliers',
       'intra_bill', 'intra_assets', 'inter_out_suplliers', 'inter_out_bill',
       'inter_out_asset', 'inter_in_suplliers', 'inter_in_bill',
       'inter_in_asset']


bill[actual_numeric].describe().round(2).T[['min','50%','max','std','mean']].rename({'50%':'median'},axis = 1)

,min,median,max,std,mean
intra_suplliers,1.0,12568.0,224042.0,47406.0,36369.02
intra_bill,1.0,590063.0,15722516.0,2306000.25,1546737.6
intra_assets,0.01,12072.31,351408.1,48015.05,32294.78
inter_out_suplliers,16.0,8972.0,168901.0,39082.95,28973.57
inter_out_bill,47.0,277976.0,8649985.0,1395652.33,882636.52
inter_out_asset,2.94,15483.23,254765.99,40543.95,30321.54
inter_in_suplliers,429.0,30387.0,259235.0,57199.34,54159.1
inter_in_bill,3641.0,494553.0,5399958.0,1010096.71,878608.93
inter_in_asset,107.81,17083.62,259141.83,38933.85,31861.57


---
Problem of 0 values in data is fixed

---

`Is there any unreal month or year value?`

In [27]:
bill['month'].unique()

array([ 7,  8,  9, 10, 11, 12,  1,  2,  3,  4,  5,  6])

In [28]:
bill['year'].unique()

array([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])

---
No unnatural year or month values found

---

`Number of months in each year`

In [29]:
bill.groupby('year')['month'].nunique().reset_index(name = 'months_available')

,year,months_available
0,2018,6
1,2019,12
2,2020,12
3,2021,12
4,2022,12
5,2023,12
6,2024,12
7,2025,12
8,2026,7


---
Keeping the incomplete months, it will help in forecasting

---

#### **GSDP Table**

In [30]:
gsdp.head()

,state/ut,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24
0,Andhra Pradesh,873721.106301,925839.118118,978581.462125,1.148471e+06,1.303524e+06,1439673.87
1,Arunachal Pradesh,25334.870000,30023.650000,30525.350000,3.477476e+04,3.962996e+04,43991.61
2,Assam,309336.320000,346850.680000,339802.980000,4.114538e+05,4.931666e+05,565400.58
3,Bihar,527975.819958,581855.484686,567262.504733,6.503024e+05,7.513956e+05,854429.00
4,Chhattisgarh,327106.660000,344672.040000,352327.510000,4.105249e+05,4.643990e+05,505886.51


In [31]:
gsdp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   state/ut  33 non-null     object 
 1   2018-19   33 non-null     float64
 2   2019-20   33 non-null     float64
 3   2020-21   33 non-null     float64
 4   2021-22   33 non-null     float64
 5   2022-23   33 non-null     float64
 6   2023-24   33 non-null     float64
dtypes: float64(6), object(1)
memory usage: 1.9+ KB


In [32]:
gsdp.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
2018-19,33.0,572711.11,618378.92,9003.18,40138.87,327106.66,857427.15,2528854.30
2019-20,33.0,614583.56,660859.80,9765.00,43421.43,346850.68,927855.00,2657370.96
2020-21,33.0,611094.44,659195.16,9310.40,39415.91,352327.51,946627.93,2627541.90
2021-22,33.0,722193.14,780994.90,10370.52,46095.74,411453.80,1124204.19,3108021.87
2022-23,33.0,828354.18,903916.13,11976.68,54284.73,493166.61,1303523.62,3645884.00
2023-24,33.0,920729.59,1005381.12,12499.11,62535.64,565400.58,1439673.87,4044251.00


---
Nothing fishy found in gsdp table

---

#### **Population Table**

In [33]:
population.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
2018,37.0,35640.00,47090.21,67.0,1489.0,19435.0,51938.0,222015.0
2019,37.0,36024.27,47657.64,68.0,1504.0,19814.0,52221.0,224979.0
2020,37.0,36408.70,48225.98,68.0,1537.0,20193.0,52504.0,227943.0
2021,37.0,36793.00,48794.89,68.0,1559.0,20571.0,52787.0,230907.0
2022,37.0,37128.68,49277.99,68.0,1567.0,20965.0,52972.0,233297.0
2023,37.0,37464.35,49761.93,69.0,1575.0,21359.0,53156.0,235687.0
2024,37.0,37799.97,50246.75,69.0,1583.0,21752.0,53340.0,238078.0
2025,37.0,38135.57,50732.19,69.0,1591.0,22146.0,53524.0,240468.0
2026,37.0,38471.27,51218.52,70.0,1604.0,22540.0,53709.0,242859.0


`Switching rows and columns of population table`

In [34]:
population = population.melt(id_vars='state/ut',var_name='year',value_name='population')

In [35]:
population.head()

,state/ut,year,population
0,Jammu & Kashmir*(UT),2018,13101
1,Himachal Pradesh,2018,7253
2,Punjab,2018,29619
3,Haryana,2018,28266
4,NCT of Delhi*,2018,19435


In [36]:
population.describe()

,population
count,333.000000
mean,37096.201201
std,48652.342776
min,67.000000
25%,1559.000000
50%,20965.000000
75%,53340.000000
max,242859.000000


### **Standardization - Existing Columns**

#### **Bill Table**

In [37]:
bill['state_name'].unique()

array(['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR',
       'CHANDIGARH', 'CHHATTISGARH', 'DADRA NAGAR HAVELI & DAMAN DIU',
       'DELHI', 'GOA', 'GUJARAT', 'HARYANA', 'HIMACHAL PRADESH',
       'JAMMU AND KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA',
       'MADHYA PRADESH', 'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM',
       'NAGALAND', 'ODISHA', 'PUDUCHERRY', 'PUNJAB', 'RAJASTHAN',
       'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH',
       'UTTARAKHAND', 'WEST BENGAL'], dtype=object)

`Lower Casing names for standardization`

In [38]:
bill['state_name'] = bill['state_name'].apply(lambda x: x.lower())

In [39]:
bill.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3201 entries, 97 to 3566
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   state_name           3201 non-null   object        
 1   year                 3201 non-null   int64         
 2   month                3201 non-null   int64         
 3   intra_suplliers      3201 non-null   Int64         
 4   intra_bill           3201 non-null   Int64         
 5   intra_assets         3201 non-null   float64       
 6   inter_out_suplliers  3201 non-null   int64         
 7   inter_out_bill       3201 non-null   int64         
 8   inter_out_asset      3201 non-null   float64       
 9   inter_in_suplliers   3201 non-null   int64         
 10  inter_in_bill        3201 non-null   int64         
 11  inter_in_asset       3201 non-null   float64       
 12  date                 3201 non-null   datetime64[ns]
dtypes: Int64(2), datetime64[ns](1), float

In [40]:
bill.head()

,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset,date
97,andhra pradesh,2018,7,25756,981997,18334.866505,16282,362407,23846.016793,53563,587192,24182.188183,2018-07-01
98,andhra pradesh,2018,8,27138,1089732,19676.725423,18129,389635,26396.241733,61724,735434,29048.179448,2018-08-01
99,andhra pradesh,2018,9,26943,1093687,19955.528909,17823,384081,25825.625389,61042,678325,27677.554892,2018-09-01
100,andhra pradesh,2018,10,28650,1149306,20449.757896,18505,379176,26534.805685,63461,730942,28944.021139,2018-10-01
101,andhra pradesh,2018,11,29398,1106203,20288.648910,18322,357570,25508.188618,62611,717828,26801.011252,2018-11-01


`Removing extra date column`

In [41]:
bill.drop(columns=['date'],inplace = True)

`Rounding off numericals`

In [42]:
bill[actual_numeric] = bill[actual_numeric].round(2)

In [43]:
bill.head()

,state_name,year,month,intra_suplliers,intra_bill,intra_assets,inter_out_suplliers,inter_out_bill,inter_out_asset,inter_in_suplliers,inter_in_bill,inter_in_asset
97,andhra pradesh,2018,7,25756,981997,18334.87,16282,362407,23846.02,53563,587192,24182.19
98,andhra pradesh,2018,8,27138,1089732,19676.73,18129,389635,26396.24,61724,735434,29048.18
99,andhra pradesh,2018,9,26943,1093687,19955.53,17823,384081,25825.63,61042,678325,27677.55
100,andhra pradesh,2018,10,28650,1149306,20449.76,18505,379176,26534.81,63461,730942,28944.02
101,andhra pradesh,2018,11,29398,1106203,20288.65,18322,357570,25508.19,62611,717828,26801.01


`Renaming Columns`

In [44]:
rename_dict = {
    'state_name': 'state',
    'intra_suplliers': 'intra_sup',
    'intra_bill': 'intra_bill',
    'intra_assets': 'intra_asset',
    'inter_out_suplliers': 'out_sup',
    'inter_out_bill': 'out_bill',
    'inter_out_asset': 'out_asset',
    'inter_in_suplliers': 'in_sup',
    'inter_in_bill': 'in_bill',
    'inter_in_asset': 'in_asset'
}

bill = bill.rename(columns=rename_dict)

In [45]:
bill.head()

,state,year,month,intra_sup,intra_bill,intra_asset,out_sup,out_bill,out_asset,in_sup,in_bill,in_asset
97,andhra pradesh,2018,7,25756,981997,18334.87,16282,362407,23846.02,53563,587192,24182.19
98,andhra pradesh,2018,8,27138,1089732,19676.73,18129,389635,26396.24,61724,735434,29048.18
99,andhra pradesh,2018,9,26943,1093687,19955.53,17823,384081,25825.63,61042,678325,27677.55
100,andhra pradesh,2018,10,28650,1149306,20449.76,18505,379176,26534.81,63461,730942,28944.02
101,andhra pradesh,2018,11,29398,1106203,20288.65,18322,357570,25508.19,62611,717828,26801.01


`Basic Sanity`

In [46]:
bill.isnull().sum()

state          0
year           0
month          0
intra_sup      0
intra_bill     0
intra_asset    0
out_sup        0
out_bill       0
out_asset      0
in_sup         0
in_bill        0
in_asset       0
dtype: int64

In [47]:
bill.duplicated().sum()

np.int64(0)

In [48]:
bill.head()

,state,year,month,intra_sup,intra_bill,intra_asset,out_sup,out_bill,out_asset,in_sup,in_bill,in_asset
97,andhra pradesh,2018,7,25756,981997,18334.87,16282,362407,23846.02,53563,587192,24182.19
98,andhra pradesh,2018,8,27138,1089732,19676.73,18129,389635,26396.24,61724,735434,29048.18
99,andhra pradesh,2018,9,26943,1093687,19955.53,17823,384081,25825.63,61042,678325,27677.55
100,andhra pradesh,2018,10,28650,1149306,20449.76,18505,379176,26534.81,63461,730942,28944.02
101,andhra pradesh,2018,11,29398,1106203,20288.65,18322,357570,25508.19,62611,717828,26801.01


`Adding Financial Year Start column to standardize with gsdp table`

In [49]:
bill['fy_start'] = np.where(bill['month']>=4, bill['year'],bill['year']-1)

In [50]:
bill.head()

,state,year,month,intra_sup,intra_bill,intra_asset,out_sup,out_bill,out_asset,in_sup,in_bill,in_asset,fy_start
97,andhra pradesh,2018,7,25756,981997,18334.87,16282,362407,23846.02,53563,587192,24182.19,2018
98,andhra pradesh,2018,8,27138,1089732,19676.73,18129,389635,26396.24,61724,735434,29048.18,2018
99,andhra pradesh,2018,9,26943,1093687,19955.53,17823,384081,25825.63,61042,678325,27677.55,2018
100,andhra pradesh,2018,10,28650,1149306,20449.76,18505,379176,26534.81,63461,730942,28944.02,2018
101,andhra pradesh,2018,11,29398,1106203,20288.65,18322,357570,25508.19,62611,717828,26801.01,2018


`GSDP do not have dadar and nagar haveli state, so we should remove that from this table too`

In [51]:
bill.state.unique()

array(['andhra pradesh', 'arunachal pradesh', 'assam', 'bihar',
       'chandigarh', 'chhattisgarh', 'dadra nagar haveli & daman diu',
       'delhi', 'goa', 'gujarat', 'haryana', 'himachal pradesh',
       'jammu and kashmir', 'jharkhand', 'karnataka', 'kerala',
       'madhya pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram',
       'nagaland', 'odisha', 'puducherry', 'punjab', 'rajasthan',
       'sikkim', 'tamil nadu', 'telangana', 'tripura', 'uttar pradesh',
       'uttarakhand', 'west bengal'], dtype=object)

In [52]:
bill = bill[bill['state'] != 'dadra nagar haveli & daman diu']

In [53]:
rename_dict = {
    'intra_asset':'intra_asset(c)',
    'out_asset':'out_asset(c)',
    'in_asset':'in_asset(c)'
}

bill = bill.rename(columns=rename_dict)

`Exporting to clean CSV`

In [54]:
out_path = os.path.join(os.getenv('OUTPUT_DIR'), 'bill_clean.csv')

bill.to_csv(out_path,index = False)

#### **GSDP Table**

In [55]:
gsdp.rename(columns={'state/ut':'state'},inplace=True)

In [56]:
gsdp.head()

,state,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24
0,Andhra Pradesh,873721.106301,925839.118118,978581.462125,1.148471e+06,1.303524e+06,1439673.87
1,Arunachal Pradesh,25334.870000,30023.650000,30525.350000,3.477476e+04,3.962996e+04,43991.61
2,Assam,309336.320000,346850.680000,339802.980000,4.114538e+05,4.931666e+05,565400.58
3,Bihar,527975.819958,581855.484686,567262.504733,6.503024e+05,7.513956e+05,854429.00
4,Chhattisgarh,327106.660000,344672.040000,352327.510000,4.105249e+05,4.643990e+05,505886.51


`Creating the financial year start column`

In [57]:
gsdp_numeric_cols = gsdp.select_dtypes(include='number').columns

In [58]:
gsdp[gsdp_numeric_cols] = gsdp[gsdp_numeric_cols].round(2)

In [59]:
gsdp = gsdp.melt(id_vars='state', var_name='fy_start', value_name='gsdp')
gsdp['fy_start'] = gsdp['fy_start'].str[:4].astype(int)

In [60]:
gsdp.sample(5)

,state,fy_start,gsdp
90,Tripura,2020,53504.12
121,Tamil Nadu,2021,2071286.16
178,Maharashtra,2023,4044251.00
114,Meghalaya,2021,38784.70
60,West Bengal,2019,1179127.46


`State name standardizations`

In [61]:
gsdp['state'].unique()

array(['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar',
       'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh',
       'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh',
       'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland',
       'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu',
       'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand',
       'West Bengal', 'Andaman & Nicobar Islands', 'Chandigarh', 'Delhi',
       'Jammu & Kashmir-UT', 'Puducherry'], dtype=object)

In [62]:
gsdp = gsdp[gsdp['state'] != 'Andaman & Nicobar Islands']

In [63]:
gsdp['state'] = gsdp['state'].str.lower().str.replace('-ut', '').str.replace('&', 'and').str.strip()

In [64]:
gsdp.head()

,state,fy_start,gsdp
0,andhra pradesh,2018,873721.11
1,arunachal pradesh,2018,25334.87
2,assam,2018,309336.32
3,bihar,2018,527975.82
4,chhattisgarh,2018,327106.66


In [65]:
gsdp = gsdp.rename(columns = {'gsdp':'gsdp(c)'})

`Exporting the data to clean table`

In [66]:
out_path = os.path.join(os.getenv('OUTPUT_DIR'), 'gsdp_clean.csv')

gsdp.to_csv(out_path,index = False)

#### **Population Table**

In [67]:
population.head()

,state/ut,year,population
0,Jammu & Kashmir*(UT),2018,13101
1,Himachal Pradesh,2018,7253
2,Punjab,2018,29619
3,Haryana,2018,28266
4,NCT of Delhi*,2018,19435


In [68]:
population.rename(columns = {'state/ut':'state'},inplace=True)

In [69]:
population.head()

,state,year,population
0,Jammu & Kashmir*(UT),2018,13101
1,Himachal Pradesh,2018,7253
2,Punjab,2018,29619
3,Haryana,2018,28266
4,NCT of Delhi*,2018,19435


`Removing the * from union territory names`

In [70]:
population['state']=population['state'].str.replace("*","").str.lower().str.replace('(ut)',"")

In [71]:
population['state'].unique()

array(['jammu & kashmir', 'himachal pradesh', 'punjab', 'haryana',
       'nct of delhi', 'rajasthan', 'uttar pradesh', 'bihar', 'assam',
       'west bengal', 'jharkhand', 'odisha', 'chhattisgarh',
       'madhya pradesh', 'gujarat', 'maharashtra', 'andhra pradesh',
       'karnataka', 'kerala', 'tamil nadu', 'chandigarh', 'uttarakhand',
       'sikkim', 'arunachal pradesh', 'nagaland', 'manipur', 'mizoram',
       'tripura', 'meghalaya', 'dadra & nagar haveli', 'daman & diu',
       'goa', 'andaman & nicobar islands', 'lakshadweep', 'puducherry',
       'telangana', 'ladakh'], dtype=object)

`renaming delhi`

In [72]:
population['state'] = population['state'].str.replace('nct of delhi','delhi')

In [73]:
population['state'].unique()

array(['jammu & kashmir', 'himachal pradesh', 'punjab', 'haryana',
       'delhi', 'rajasthan', 'uttar pradesh', 'bihar', 'assam',
       'west bengal', 'jharkhand', 'odisha', 'chhattisgarh',
       'madhya pradesh', 'gujarat', 'maharashtra', 'andhra pradesh',
       'karnataka', 'kerala', 'tamil nadu', 'chandigarh', 'uttarakhand',
       'sikkim', 'arunachal pradesh', 'nagaland', 'manipur', 'mizoram',
       'tripura', 'meghalaya', 'dadra & nagar haveli', 'daman & diu',
       'goa', 'andaman & nicobar islands', 'lakshadweep', 'puducherry',
       'telangana', 'ladakh'], dtype=object)

`Removing extra states for standardization of all three tables`

In [74]:
to_be_removed = ['andaman & nicobar islands', 'dadra & nagar haveli', 'daman & diu', 'ladakh', 'lakshadweep']

population = population[~population['state'].isin(to_be_removed)]

#### **All 3 Tables**

`Checking extra states names in all 3 tables`

In [75]:
def clean(s):
    return s.astype(str).str.strip().str.lower().str.replace("&", "and").str.replace(r"\s+", " ", regex=True)

gsdp_s = set(clean(gsdp["state"]))
population_s  = set(clean(population["state"]))
bill_s = set(clean(bill["state"]))

all_s = gsdp_s | population_s | bill_s
common = gsdp_s & population_s & bill_s

print("in all 3:", len(common))
print("gsdp only:", sorted(gsdp_s - population_s - bill_s))
print("pop only:", sorted(population_s - gsdp_s - bill_s))
print("bill only:", sorted(bill_s - gsdp_s - population_s))

# names missing from each table
print("missing in gsdp:", sorted(all_s - gsdp_s))
print("missing in pop:", sorted(all_s - population_s))
print("missing in bill:", sorted(all_s - bill_s))

in all 3: 32
gsdp only: []
pop only: []
bill only: []
missing in gsdp: []
missing in pop: []
missing in bill: []


In [76]:
population.head()

,state,year,population
0,jammu & kashmir,2018,13101
1,himachal pradesh,2018,7253
2,punjab,2018,29619
3,haryana,2018,28266
4,delhi,2018,19435


In [77]:
population.rename(columns = {'population':'population(c)'},inplace=True)

`Exporting the cleaned data`

In [78]:
out_path = os.path.join(os.getenv('OUTPUT_DIR'), 'population_clean.csv')

population.to_csv(out_path,index = False)